In [1]:
import requests
from bs4 import BeautifulSoup

url = "https://globalsummitguide.com/master-mountain-list"
response = requests.get(url)
soup = BeautifulSoup(response.content, 'html.parser')

raw_names = []
mountains_list = soup.find_all('td', class_=None)

# find names in table (in span or a)
for td in mountains_list:
    a_tag = td.find('a')
    span_tag = td.find('span', class_='mtn-plain')
    
    if a_tag:
        raw_names.append(a_tag.get_text(strip=True))
    elif span_tag:
        raw_names.append(span_tag.get_text(strip=True))

# сlean from text in brackets
clean_names = []
for name in raw_names:
    if "(" in name:
        name = name.split("(")[0].strip()
    if name and name not in clean_names:
        clean_names.append(name)

print(f"Amount of collected mountains: {len(clean_names)}")
print(clean_names[:5])

Amount of collected mountains: 498
['Mount Everest', 'K2', 'Kangchenjunga', 'Lhotse', 'Makalu']


In [2]:
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path=".env", override=True)

openapi_key=os.getenv("AI_API")



In [9]:
import random

# pool of structural styles to force syntactic diversity
style_pool = [
    "use passive voice at least once",
    "write one sentence as a direct quote from a person",
    "make one sentence look like an old historical record or archive text",
    "include other geographic features like rivers, valleys, or cities in the same text",
    "use complex punctuation (dashes, semicolons, or parenthetical remarks)",
    "start at least one sentence with a subordinate clause (e.g., 'although...', 'while...')",
    "make one sentence look like a technical scientific report about geology",
    "ensure the mountain name appears in the exact middle or end of the text, not just the beginning",
]

# list of corporate environments for negatives
negative_contexts = [
    "gaming/video game titles",
    "luxury watch and fountain pen manufacturing",
    "venture capital, asset management, and investment funds",
    "streetwear, outdoor clothing lines, and fashion brands",
    "cloud computing, infrastructure software, and database tech startups",
    "local artisanal cafes, bistros, and roasteries",
]

def get_dynamic_prompt(mountain_batch):
    mountains_str = ", ".join(mountain_batch)

    # pick random constraints for this specific batch loop
    selected_styles = random.sample(style_pool, k=2)
    selected_negatives = random.sample(negative_contexts, k=2)

    # assemble the ultimate prompt
    prompt = f"""
    act as an expert computational linguist and gold-standard ner data annotator.
    your task is to generate a highly diverse, non-templated dataset for named entity recognition (class: MOUNTAIN).
    
    target entities for this batch: [{mountains_str}]
    
    generation requirements:
    1. for each entity, generate exactly 2 positive sentences (where it is a real mountain) and 2 hard negative sentences (where it is NOT a mountain).
    2. structural diversity constraints for this batch (apply them across your answers):
       - {selected_styles[0]}
       - {selected_styles[1]}
    3. negative context constraints for this batch:
       - use these specific domains for corporate/product naming traps: {selected_negatives[0]} or {selected_negatives[1]}.
    4. linguistic rule: strictly avoid generic clichés like "the view from the base camp is breathtaking", "reached the summit at dawn", or simply "[mountain] logistics". make sentences natural, complex, and human-written.
    5. you are absolutely forbidden to write anything other than the json of the response, neither at the beginning nor at the end of the response, you start and end your response exclusively with a dataset without any unnecessary phrases.
    
    return only a valid json object with a single key "data" that contains the array.
    do not wrap the response in markdown code fences.
    expected json structure:
    {{
      "data": [
        {{
          "text": "sentence content here",
          "entities": [{{"start": int, "end": int, "label": "MOUNTAIN"}}]
        }}
      ]
    }}
    """
    return prompt

In [10]:
import json
import google.generativeai as gemini

gemini.configure(api_key=openapi_key)


def generate_dataset_from_chatgpt(mountain_batch):
    mountains_str = ", ".join(mountain_batch)
    prompt = get_dynamic_prompt(mountains_str)

    try:
        model = gemini.GenerativeModel(
            "gemini-3.1-flash-lite",
            generation_config={"response_mime_type": "application/json"},
        )
        response = model.generate_content(prompt)
        raw_json = json.loads(response.text)

        # safely extract the target list from our new data root key
        if "data" in raw_json and isinstance(raw_json["data"], list):
            return raw_json["data"]

        # backup recovery path just in case the key name variance happens
        for key in raw_json.keys():
            if isinstance(raw_json[key], list):
                return raw_json[key]

        print(f"expected data list key missing for batch: {mountain_batch}")
        return []

    except Exception as e:
        # no more parsing crashes, handles rate limits or network drops gracefully
        print(f"failed batch {mountain_batch}: {e}")
        return []

In [11]:
import json
import re
import google.generativeai as gemini

gemini.configure(api_key=openapi_key)


def generate_dataset_from_chatgpt(mountain_batch):
    mountains_str = ", ".join(mountain_batch)
    prompt = get_dynamic_prompt(mountains_str)

    try:
        model = gemini.GenerativeModel(
            "gemini-3.1-flash-lite",
            generation_config={"response_mime_type": "application/json"},
        )
        response = model.generate_content(prompt)

        # fix: extract the raw string from the gemini response object
        raw_text = response.text

        # now passing the text string to regex instead of the object
        json_match = re.search(r"\[.*\]", raw_text, re.DOTALL)

        if json_match:
            clean_json_str = json_match.group(0)
            return json.loads(clean_json_str)

        # fallback path if model returned a top-level object {} instead of array
        dict_match = re.search(r"\{.*\}", raw_text, re.DOTALL)
        if dict_match:
            data = json.loads(dict_match.group(0))
            # try to grab the first available list inside the dict keys
            for key in data.keys():
                if isinstance(data[key], list):
                    return data[key]

        print(
            f"could not find any valid json block in output for: {mountain_batch}"
        )
        return []

    except Exception as e:
        print(f"failed batch {mountain_batch}: {e}")
        print(f"response: {response}")
        return []

In [12]:
dataset = []

for i in range(0, 355, 6):
    answer=generate_dataset_from_chatgpt(clean_names[i:i+6])
    dataset.extend(answer)

failed batch ['Mount Kazbek', 'Khan Tengri', 'Jengish Chokusu', 'Peak Lenin', 'Ismoil Somoni Peak', 'Peak Korzhenevskaya']: Extra data: line 27 column 1 (char 4559)
response: response:
GenerateContentResponse(
    done=True,
    iterator=None,
    result=protos.GenerateContentResponse({
      "candidates": [
        {
          "content": {
            "parts": [
              {
                "text": "{\n  \"data\": [\n    {\"text\": \"Geological surveys indicate that Mount Kazbek remains one of the most volatile volcanic systems in the Caucasus region.\", \"entities\": [{\"start\": 21, \"end\": 33, \"label\": \"MOUNTAIN\"}]},\n    {\"text\": \"Heavy snowfall (often exceeding three meters) has been recorded at the slopes of Mount Kazbek during the transition into autumn.\", \"entities\": [{\"start\": 57, \"end\": 69, \"label\": \"MOUNTAIN\"}]},\n    {\"text\": \"The latest Kubernetes deployment strategy utilized by Kazbek Cloud Services simplifies cross-cluster networking for enterpr

In [13]:
import json 

with open("dataset.json", "w", encoding="utf-8") as file:
    json.dump(dataset, file, ensure_ascii=False, indent=2)

In [19]:
import json

# load your current dataset file
with open("dataset.json", "r", encoding="utf-8") as f:
    dataset = json.load(f)


# sort by length desc to prevent matching "Everest" inside "Mount Everest"
clean_names.sort(key=len, reverse=True)

fixed_dataset = []

for item in dataset:
    text = item["text"]
    entities = item["entities"]

    # if entities is empty it is a validated hard negative, save it directly
    if not entities:
        fixed_dataset.append(item)
        continue

    new_entities = []
    for mtn in clean_names:
        start_idx = text.find(mtn)

        if start_idx != -1:
            end_idx = start_idx + len(mtn)
            new_entities.append(
                {"start": start_idx, "end": end_idx, "label": "MOUNTAIN"}
            )
            break

    if new_entities:
        item["entities"] = new_entities
        fixed_dataset.append(item)
    else:
        print(f"skipping item without active match: {text}")
        
with open("dataset.json", "w", encoding="utf-8") as f:
    json.dump(fixed_dataset, f, ensure_ascii=False, indent=2)

print(f"alignment patch complete! clean records saved: {len(fixed_dataset)}")

KeyError: 'entities'

In [20]:
# load current file to check the slices
with open("dataset.json", "r", encoding="utf-8") as f:
    dataset = json.load(f)

print("--- validation report ---")
for i, item in enumerate(dataset):
    text = item["text"]
    entities = item["entities"]
    
    if entities:
        start = entities[0]["start"]
        end = entities[0]["end"]
        # pulling the actual string that lives under these indices
        actual_substring = text[start:end]

        if actual_substring not in clean_names: 
            print(f"line {i} | indices: {start}:{end} | actual text inside: '{actual_substring}'")

--- validation report ---
line 0 | indices: 57:70 | actual text inside: 'nt surge in i'
line 1 | indices: 89:102 | actual text inside: 's that Mount '
line 4 | indices: 35:37 | actual text inside: ' u'
line 5 | indices: 36:38 | actual text inside: 'as'
line 8 | indices: 40:52 | actual text inside: 'across the m'
line 9 | indices: 65:77 | actual text inside: 'rops of Kang'
line 12 | indices: 25:31 | actual text inside: ' of Lh'
line 13 | indices: 22:28 | actual text inside: 'of Lho'
line 17 | indices: 41:47 | actual text inside: 'near M'
line 20 | indices: 29:36 | actual text inside: 'of Cho '
line 21 | indices: 13:20 | actual text inside: ' Cho Oy'
line 24 | indices: 95:106 | actual text inside: 'f Dhaulagir'
line 25 | indices: 102:113 | actual text inside: 'e of Dhaula'
line 26 | indices: 57:68 | actual text inside: 'wn as Dhaul'
line 27 | indices: 72:83 | actual text inside: 'server for '
line 28 | indices: 96:103 | actual text inside: ' profil'
line 29 | indices: 112:119 | actual te

KeyError: 'entities'